# 03 — Live MC controls deep-dive

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/03_live_controls.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/03_live_controls.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/03_live_controls.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Quantify how the Live-tab knobs (N, window, stride, seed) affect the resulting statistics — including a √N-rule sanity check.

**Mirrors.** **Live MC** tab → *Runs N* · *Window* · *Conv. stride* · *Seed*.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. The five Live-tab knobs

| Knob | What it does | This notebook varies |
|---|---|---|
| **N**           | total Monte-Carlo runs | 1k → 200k |
| **Rate**        | streaming speed (UI only — does not affect math) | unused here |
| **Window**      | trailing window for the streaming statistics | 50 / 200 / 1000 |
| **Conv. stride**| how often a CI on the mean is recomputed | 25 / 100 |
| **Seed**        | mulberry32 PRNG seed | 42 / 1337 / 2025 |


In [ ]:
import numpy as np, pandas as pd
from cubespec import DEFAULT_CSP, sample_independent, compute_outputs_batch, bootstrap_mean_ci

rows = []
for N in (1_000, 5_000, 20_000, 100_000):
    Y = compute_outputs_batch(sample_independent(DEFAULT_CSP, n=N, seed=1337))[:, 2]
    ci = bootstrap_mean_ci(Y, B=400, seed=1337)
    rows.append({'N': N, 'mean': Y.mean(), 'sd': Y.std(ddof=1),
                 'ci_lo': ci.lo, 'ci_hi': ci.hi, 'ci_width': ci.hi - ci.lo})
df_n = pd.DataFrame(rows)
df_n


The CI width should shrink ≈ 1/√N — verify the ratios:


In [ ]:
for i in range(1, len(df_n)):
    ratio = df_n.ci_width[i-1] / df_n.ci_width[i]
    expected = float(np.sqrt(df_n.N[i] / df_n.N[i-1]))
    print(f'N {df_n.N[i-1]:>6} → {df_n.N[i]:>6}: width ratio {ratio:.2f}  (≈ √(N₂/N₁) = {expected:.2f})')


## 2. Convergence-stride: how often we draw a CI


In [ ]:
import matplotlib.pyplot as plt
Y = compute_outputs_batch(sample_independent(DEFAULT_CSP, n=20_000, seed=1337))[:, 2]
fig, ax = plt.subplots(figsize=(8, 3.6))
for stride, color in zip((25, 100, 500), ('#5b8def', '#e94f64', '#28a745')):
    ks = np.arange(stride, len(Y) + 1, stride)
    means = np.array([Y[:k].mean() for k in ks])
    ax.plot(ks, means, color=color, lw=1.5, label=f'stride = {stride}')
ax.axhline(Y.mean(), color='black', lw=0.8, ls='--', alpha=0.6, label='final mean')
ax.set_xlabel('runs accumulated')
ax.set_ylabel('running mean of P9 (MPa)')
ax.set_title('Convergence stride only changes how often we sample, not the underlying behaviour')
ax.legend(); fig.tight_layout(); plt.show()


## 3. Seed reproducibility

Same seed must return bit-identical arrays. Different seeds must agree on the mean to within the CI, but not exactly.


In [ ]:
Y_a = compute_outputs_batch(sample_independent(DEFAULT_CSP, n=20_000, seed=1337))[:, 2]
Y_b = compute_outputs_batch(sample_independent(DEFAULT_CSP, n=20_000, seed=1337))[:, 2]
Y_c = compute_outputs_batch(sample_independent(DEFAULT_CSP, n=20_000, seed=2025))[:, 2]
print('seed 1337 ↔ 1337 identical :', np.array_equal(Y_a, Y_b))
print(f'seed 1337 mean = {Y_a.mean():.4f}')
print(f'seed 2025 mean = {Y_c.mean():.4f}')
print(f'|Δmean|        = {abs(Y_a.mean() - Y_c.mean()):.4f} MPa')


## 4. Try this

In the dashboard, set N = 5 000, stride = 25 and watch the convergence panel. Then change stride to 500 — the line gets coarser without the underlying numbers changing. The notebook above shows the same effect.


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
